# HDD 16 — Matched Q3 detector and SMART-baseline comparison

This notebook compares Notebook 15's Isolation Forest and Notebook 14's two SMART baseline scores on **exactly the same held-out drives and September endpoints**. It preserves the original fixed `>0` baseline rules and also shows separately calibrated 99.8th-percentile baseline rules, using Notebook 15's *calibration drives* from August 16–31. No threshold is fitted to failure labels. The comparison is exploratory: aggregate Q3 outcomes were already viewed, and Q4 stays closed.

**Expected:** 30-day AUPRC and false-alert rates may remain undefined when September has no complete 30-day negative follow-up. Strict event recall may still be measurable on complete pre-failure histories. Undefined values remain missing.

In [4]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib,json,zipfile,re
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score
try:
    from IPython.display import display
except ImportError:
    display=print
COMPONENT_ROOT=Path(r'D:/SLIIT/Y4S1/RP/Research/J26-DS-342-/01_component_1_behavioral_drift_genai')
RUN_ID='single_model_q1q2_2021_v1'
AUDIT_ID='q3_multimodel_event_coverage_v1'
BASE_ID='q3_multimodel_warning_baselines_v1'
PILOT_ID='q3_five_model_isolation_forest_v1'
COMPARE_ID='q3_matched_detector_baseline_comparison_v1'
AUDIT=COMPONENT_ROOT/'results'/RUN_ID/AUDIT_ID
BASE=COMPONENT_ROOT/'results'/RUN_ID/BASE_ID
PILOT=COMPONENT_ROOT/'results'/RUN_ID/PILOT_ID
OUT=COMPONENT_ROOT/'results'/RUN_ID/COMPARE_ID
OUT.mkdir(parents=True,exist_ok=True)
Q3_ZIP=COMPONENT_ROOT/'data/raw/data_Q3_2021.zip'
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(4*1024*1024),b''):h.update(chunk)
    return h.hexdigest()
def load_json(path):
    assert path.is_file(),f'Missing input: {path}'
    return json.loads(path.read_text())
up=load_json(PILOT/'pilot_complete.json')
assert up['analysis_complete'] and not up['future_test_outcomes_opened'] and not up['sealed_June_test_opened']
audit=load_json(AUDIT/'multimodel_coverage_complete.json')
base=load_json(BASE/'baseline_evaluation_complete.json')
assert audit['analysis_complete'] and base['analysis_complete']
reservation_path=COMPONENT_ROOT/audit['split_path']
assert sha256(reservation_path)==audit['split_sha256']
reservation=load_json(reservation_path)
assert sha256(Q3_ZIP)==audit['settings']['archive_sha256']
models=list(reservation['models']);assert len(models)==5
pilot_settings=load_json(PILOT/'settings.json')
assert pilot_settings['evaluation_metric_dates']==['2021-09-01','2021-09-30']
assert pilot_settings['calibration_dates']==['2021-08-16','2021-08-31']
assert pilot_settings['calibration_quantile']==.998
assert sha256(PILOT/'settings.json')==up['settings_sha256']
score_manifest=load_json(PILOT/'score_manifest.json')
assert sha256(PILOT/'score_manifest.json')==up['score_manifest_sha256']
base_manifest=load_json(BASE/'score_manifest.json')
for name,digest in score_manifest['score_sha256'].items():
    assert sha256(PILOT/'score_partitions'/name)==digest
for name,digest in base_manifest['partition_sha256'].items():
    assert sha256(BASE/'daily_scores'/name)==digest
SALT=pilot_settings['salt']
def group_for(serial):
    x=int(hashlib.sha256((SALT+'|'+str(serial)).encode()).hexdigest()[:16],16)/16**16
    return 'fit' if x<.60 else ('calibration' if x<.80 else 'evaluation')
role={str(s):group_for(s) for m,g in reservation['models'].items() for s in g['development']}
model_for={str(s):m for m,g in reservation['models'].items() for s in g['development']}
reserved={str(s) for g in reservation['models'].values() for s in g['future_test']}
assert set(role).isdisjoint(reserved)
DATES=pd.date_range('2021-07-01','2021-09-30')
DAY={str(x.date()):i for i,x in enumerate(DATES)}
protocol={'pilot_score_manifest_sha256':sha256(PILOT/'score_manifest.json'),
          'baseline_score_manifest_sha256':sha256(BASE/'score_manifest.json'),
          'reservation_sha256':sha256(reservation_path),'q3_zip_sha256':sha256(Q3_ZIP),
          'population':'Notebook 15 evaluation drives and their scored Aug-Sep endpoints',
          'endpoint_metric_dates':['2021-09-01','2021-09-30'],
          'event_dates':['2021-09-01','2021-09-30'],
          'horizons_days':[7,14,30],
          'fixed_baseline_rule':'score > 0',
          'calibrated_baseline_rule':'score > per-model higher 0.998 quantile on distinct calibration drives Aug 16-31',
          'negative_rule':'every following calendar day observed within Q3 and no failure in horizon',
          'event_rule':'all h pre-failure endpoints have both detector and baseline score',
          'winner_selection_enabled':False,'test_outcomes_opened':False}
p=OUT/'comparison_protocol.json'
if p.exists():assert load_json(p)==protocol,'Protocol changed: preserve old comparison or use a new ID.'
else:p.write_text(json.dumps(protocol,indent=2))
print('Upstream score hashes verified. Matched evaluation drives:',sum(v=='evaluation' for v in role.values()))

Upstream score hashes verified. Matched evaluation drives: 15589


## 1. Match the frozen score rows and fit optional baseline thresholds

The fixed SMART baselines use their original `score > 0` rule. The optional calibrated versions apply the *same 0.998 calibration quantile policy* as Isolation Forest, but matching a calibration quantile does **not guarantee identical alert burden** on September. Every later metric uses exact drive/date joins. Save thresholds and the matched score table before reading events.

In [5]:
pilot_rows=[];baseline_rows=[];cal_rows=[]
for date in DAY:
    if date<'2021-08-01':continue
    pn=PILOT/'score_partitions'/f'{date}.parquet'
    bn=BASE/'daily_scores'/f'{date}.parquet'
    a=pd.read_parquet(pn,columns=['serial_number','model','end_date','score','threshold','is_anomaly_candidate'])
    b=pd.read_parquet(bn,columns=['serial_number','model','end_date','sector_positive','sector_delta'])
    if '2021-08-16'<=date<='2021-08-31':
        cal=b.loc[b.serial_number.map(lambda s:role.get(str(s))).eq('calibration')].copy()
        cal_rows.append(cal)
    ev=b.loc[b.serial_number.map(lambda s:role.get(str(s))).eq('evaluation')].copy()
    assert not a.duplicated(['serial_number','end_date']).any() and not ev.duplicated(['serial_number','end_date']).any()
    assert set(a.serial_number)==set(ev.serial_number),f'Unmatched score population on {date}'
    pilot_rows.append(a);baseline_rows.append(ev)
P=pd.concat(pilot_rows,ignore_index=True)
B=pd.concat(baseline_rows,ignore_index=True)
C=pd.concat(cal_rows,ignore_index=True)
assert not P.duplicated(['serial_number','end_date']).any()
assert not B.duplicated(['serial_number','end_date']).any()
assert not C.duplicated(['serial_number','end_date']).any()
keys=['serial_number','model','end_date']
paired=P.merge(B,on=keys,how='outer',validate='one_to_one',indicator=True)
assert paired._merge.eq('both').all() and len(paired)==len(P)==len(B)
paired.drop(columns='_merge',inplace=True)
assert set(paired.serial_number).isdisjoint(reserved)
assert all(model_for[s]==m for s,m in paired[['serial_number','model']].drop_duplicates().itertuples(index=False,name=None))
assert np.isfinite(paired[['score','threshold','sector_positive','sector_delta']].to_numpy()).all()
np.testing.assert_array_equal(paired.is_anomaly_candidate.to_numpy(dtype=bool),(paired.score>paired.threshold).to_numpy())
thresholds={}
for model in models:
    x=C.loc[C.model.eq(model)]
    assert len(x)>100 and x.serial_number.nunique()>100
    thresholds[model]={field:float(np.quantile(x[field].to_numpy(dtype=float),.998,method='higher'))
                       for field in ['sector_positive','sector_delta']}
threshold_path=OUT/'baseline_calibration_thresholds.json'
if threshold_path.exists():assert load_json(threshold_path)==thresholds
else:threshold_path.write_text(json.dumps(thresholds,indent=2))
score_path=OUT/'matched_scores.parquet'
if score_path.exists():pd.testing.assert_frame_equal(pd.read_parquet(score_path),paired)
else:paired.to_parquet(score_path,index=False)
score_lock={'protocol_sha256':sha256(p),'baseline_threshold_sha256':sha256(threshold_path),
            'matched_scores_sha256':sha256(score_path),'rows':len(paired),
            'failure_labels_used_for_scoring':False}
lock_path=OUT/'score_lock.json'
if lock_path.exists():assert load_json(lock_path)==score_lock
else:lock_path.write_text(json.dumps(score_lock,indent=2))
print('Matched score rows frozen:',len(paired))
display(pd.DataFrame(thresholds).T)

Matched score rows frozen: 945649


,sector_positive,sector_delta
TOSHIBA MG07ACA14TA,2.0,0.0
ST12000NM0008,576.0,0.0
ST4000DM000,1760.0,0.0
ST8000NM0055,2360.0,8.0
HGST HMS5C4040BLE640,96.0,0.0


## 2. Create one shared set of targets and event histories

Recorded first failures come from Notebook 13. Read only serial IDs from the Q3 ZIP to check complete follow-up for negative endpoints. Failure day and later endpoints are excluded. A positive endpoint can be established by a recorded event even if later daily rows are missing. The score table supplies the exact available warning dates.

In [6]:
events_path=COMPONENT_ROOT/'data/evaluation_only'/RUN_ID/AUDIT_ID/'development_first_failure_support.csv'
events=pd.read_csv(events_path,dtype={'serial_number':str},parse_dates=['failure_date'])
events=events.loc[events.serial_number.map(lambda s:role.get(s)).eq('evaluation')].copy()
assert events.serial_number.is_unique
assert set(events.serial_number).isdisjoint(reserved)
# Some declared evaluation drives fail before a valid scored window exists.
# Preserve those events; strict event recall will mark their history incomplete.
unscoreable=events.loc[~events.serial_number.isin(paired.serial_number)].copy()
unscoreable['reason']='no_valid_scored_window_in_matched_Aug_Sep_tables'
unscoreable.to_csv(OUT/'recorded_events_without_scored_windows.csv',index=False)
print('Recorded evaluation-drive failures without scored windows:',len(unscoreable))
first={s:DAY[str(pd.Timestamp(d).date())] for s,d in events[['serial_number','failure_date']].itertuples(index=False,name=None)}
with zipfile.ZipFile(Q3_ZIP) as z:
    members={}
    for info in z.infolist():
        if info.is_dir() or '__MACOSX' in info.filename:continue
        match=re.fullmatch(r'(2021-\d{2}-\d{2})\.csv',Path(info.filename).name)
        if match and match.group(1) in DAY:
            assert match.group(1) not in members
            members[match.group(1)]=info.filename
assert set(members)==set(DAY)
observed={}
with zipfile.ZipFile(Q3_ZIP) as z:
    for date,member in sorted(members.items()):
        seen=set();bit=1<<DAY[date]
        with z.open(member) as f:
            for chunk in pd.read_csv(f,usecols=['serial_number'],dtype={'serial_number':str},chunksize=50000):
                serials=chunk.loc[chunk.serial_number.map(lambda s:role.get(s)).eq('evaluation'),'serial_number'].tolist()
                assert len(serials)==len(set(serials)) and seen.isdisjoint(serials)
                seen.update(serials)
                for serial in serials:observed[serial]=observed.get(serial,0)|bit

def all_days(bits,lo,hi):
    if lo<0 or hi>=len(DATES):return False
    mask=((1<<(hi-lo+1))-1)<<lo
    return bits&mask==mask
sept=paired.loc[paired.end_date.ge('2021-09-01')].copy().reset_index(drop=True)
assert sept.end_date.le('2021-09-30').all()
for h in protocol['horizons_days']:
    vals=[]
    for serial,date in sept[['serial_number','end_date']].itertuples(index=False,name=None):
        day=DAY[date];failure=first.get(serial)
        if failure is not None and failure<=day:v=np.nan
        elif failure is not None and failure-day<=h:v=1
        elif all_days(observed.get(serial,0),day+1,day+h):v=0
        else:v=np.nan
        vals.append(v)
    sept[f'target_h{h}']=vals
support=[]
for h in protocol['horizons_days']:
    for model in models:
        s=sept.loc[sept.model.eq(model),f'target_h{h}']
        support.append({'model':model,'horizon_days':h,'scoreable_endpoints':len(s),
                        'positive_endpoints':int(s.eq(1).sum()),'negative_endpoints':int(s.eq(0).sum()),
                        'censored_or_post_failure':int(s.isna().sum())})
support=pd.DataFrame(support)
support.to_csv(OUT/'matched_target_support.csv',index=False)
display(support)

Recorded evaluation-drive failures without scored windows: 11


,model,horizon_days,scoreable_endpoints,positive_endpoints,negative_endpoints,censored_or_post_failure
0,TOSHIBA MG07ACA14TA,7,154890,14,118744,36132
1,ST12000NM0008,7,91357,28,70019,21310
2,ST4000DM000,7,91041,49,69739,21253
3,ST8000NM0055,7,66722,21,51159,15542
4,HGST HMS5C4040BLE640,7,60427,0,46380,14047
5,TOSHIBA MG07ACA14TA,14,154890,28,82597,72265
6,ST12000NM0008,14,91357,53,48731,42573
7,ST4000DM000,14,91041,92,48451,42498
8,ST8000NM0055,14,66722,42,35636,31044
9,HGST HMS5C4040BLE640,14,60427,0,32317,28110


## 3. Evaluate identical endpoints and complete event histories

Five rows per model/horizon are reported: Isolation Forest, two original fixed SMART rules, and two calibration-quantile SMART rules. Average precision uses the continuous score even when a threshold yields no alerts. Event recall requires all h preceding days in the matched score table. Do not read a difference as statistically reliable from only a few recorded events.

In [7]:
methods=['isolation_forest','sector_positive_fixed','sector_delta_fixed','sector_positive_calibrated','sector_delta_calibrated']
def score_flag(sub,model,method):
    if method=='isolation_forest':return sub.score.to_numpy(dtype=float),sub.is_anomaly_candidate.to_numpy(dtype=bool)
    field='sector_positive' if 'sector_positive' in method else 'sector_delta'
    score=sub[field].to_numpy(dtype=float)
    threshold=0.0 if method.endswith('fixed') else thresholds[model][field]
    return score,score>threshold
rows=[];details=[]
for model in models:
    full=paired.loc[paired.model.eq(model)].copy()
    sub=sept.loc[sept.model.eq(model)].copy()
    for method in methods:
        score,flags=score_flag(sub,model,method)
        allscore,allflags=score_flag(full,model,method)
        full['flag']=allflags
        by_serial={s:g.set_index('end_date') for s,g in full.groupby('serial_number',sort=False)}
        for h in protocol['horizons_days']:
            target=sub[f'target_h{h}'].to_numpy(dtype=float)
            keep=np.isfinite(target);y=target[keep].astype(int);s=score[keep];f=flags[keep]
            pos=int((y==1).sum());neg=int((y==0).sum())
            tp=int((f&(y==1)).sum());fp=int((f&(y==0)).sum())
            drows=[]
            for event in events.loc[events.model.eq(model)].itertuples(index=False):
                fail=pd.Timestamp(event.failure_date)
                if not pd.Timestamp('2021-09-01')<=fail<=pd.Timestamp('2021-09-30'):continue
                d=DAY[str(fail.date())]
                required=[str(DATES[k].date()) for k in range(max(0,d-h),d)]
                g=by_serial.get(event.serial_number)
                complete=bool(g is not None and len(required)==h and all(x in g.index for x in required))
                alarms=g.loc[required,'flag'] if complete else pd.Series(dtype=bool)
                first_alert=required[np.flatnonzero(alarms.to_numpy())[0]] if complete and alarms.any() else None
                drows.append({'serial_number':event.serial_number,'model':model,'method':method,
                              'horizon_days':h,'failure_date':fail,'complete_warning_history':complete,
                              'recalled':bool(complete and alarms.any()),
                              'first_warning_lead_days':(fail-pd.Timestamp(first_alert)).days if first_alert else np.nan})
            ed=pd.DataFrame(drows,columns=['serial_number','model','method','horizon_days','failure_date','complete_warning_history','recalled','first_warning_lead_days'])
            complete_events=ed.loc[ed.complete_warning_history];recalled=complete_events.loc[complete_events.recalled]
            details.append(ed)
            rows.append({'model':model,'method':method,'horizon_days':h,
                         'scored_endpoints':len(sub),'positive_endpoints':pos,'negative_endpoints':neg,
                         'censored_or_post_failure':int((~keep).sum()),
                         'average_precision':float(average_precision_score(y,s)) if pos and neg else np.nan,
                         'positive_prevalence':float(y.mean()) if len(y) else np.nan,
                         'threshold_precision':tp/(tp+fp) if tp+fp else np.nan,
                         'positive_endpoint_recall':tp/pos if pos else np.nan,
                         'false_alerts_per_1000_observed_negative_endpoints':1000*fp/neg if neg else np.nan,
                         'all_alerts_per_1000_scored_endpoints':1000*flags.mean() if len(flags) else np.nan,
                         'September_events':len(ed),'events_with_complete_warning_history':len(complete_events),
                         'events_recalled':len(recalled),
                         'strict_event_recall':len(recalled)/len(complete_events) if len(complete_events) else np.nan,
                         'median_first_warning_lead_days':float(recalled.first_warning_lead_days.median()) if len(recalled) else np.nan})
metrics=pd.DataFrame(rows)
assert len(metrics)==len(models)*len(methods)*3
for h in protocol['horizons_days']:
    reference=metrics.loc[metrics.horizon_days.eq(h)].groupby('model')[['positive_endpoints','negative_endpoints','events_with_complete_warning_history']].nunique()
    assert reference.eq(1).all().all(),'Methods do not share one evaluation population.'
metrics.to_csv(OUT/'matched_method_metrics.csv',index=False)
event_details=pd.concat(details,ignore_index=True)
event_details.to_csv(OUT/'matched_event_details.csv',index=False)
display(metrics.loc[metrics.horizon_days.eq(14),['model','method','positive_endpoints','average_precision','events_with_complete_warning_history','strict_event_recall','false_alerts_per_1000_observed_negative_endpoints','all_alerts_per_1000_scored_endpoints']])
print('Calibration quantiles are the same policy, not guaranteed equal realized alert budgets.')

C:\Users\IMAKA\AppData\Local\Temp\ipykernel_26728\1488723700.py:58: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  event_details=pd.concat(details,ignore_index=True)


,model,method,positive_endpoints,average_precision,events_with_complete_warning_history,strict_event_recall,false_alerts_per_1000_observed_negative_endpoints,all_alerts_per_1000_scored_endpoints
1,TOSHIBA MG07ACA14TA,isolation_forest,28,0.000229,2,0.000000,1.053307,1.039447
4,TOSHIBA MG07ACA14TA,sector_positive_fixed,28,0.000339,2,0.000000,7.239972,7.443993
7,TOSHIBA MG07ACA14TA,sector_delta_fixed,28,0.000339,2,0.000000,0.121070,0.142036
10,TOSHIBA MG07ACA14TA,sector_positive_calibrated,28,0.000339,2,0.000000,1.537586,1.575312
13,TOSHIBA MG07ACA14TA,sector_delta_calibrated,28,0.000339,2,0.000000,0.121070,0.142036
16,ST12000NM0008,isolation_forest,53,0.162146,4,0.500000,1.744270,2.003131
19,ST12000NM0008,sector_positive_fixed,53,0.041907,4,0.500000,13.872073,14.317458
22,ST12000NM0008,sector_delta_fixed,53,0.012885,4,0.500000,0.369375,0.317436
25,ST12000NM0008,sector_positive_calibrated,53,0.041907,4,0.250000,3.221768,3.568418
28,ST12000NM0008,sector_delta_calibrated,53,0.012885,4,0.500000,0.369375,0.317436


Calibration quantiles are the same policy, not guaranteed equal realized alert budgets.


## 4. Paired descriptive event comparison and closeout

The paired event table shows which failures each method detected on the same complete histories. It is descriptive; the event sample is small and no method is chosen from these results. Final evaluation requires a frozen model/protocol and independent Q4 data with enough follow-up.

In [8]:
paired_events=event_details.loc[event_details.complete_warning_history].pivot_table(index=['model','horizon_days','serial_number','failure_date'],columns='method',values='recalled',aggfunc='first').reset_index()
paired_events.to_csv(OUT/'paired_complete_event_outcomes.csv',index=False)
summary=[]
for h in protocol['horizons_days']:
    g=metrics.loc[metrics.horizon_days.eq(h)]
    summary.append({'horizon_days':h,'evaluated_models':len(models),
                    'models_with_undefined_AP':int(g.loc[g.method.eq('isolation_forest'),'average_precision'].isna().sum()),
                    'complete_events_across_models':int(g.loc[g.method.eq('isolation_forest'),'events_with_complete_warning_history'].sum())})
summary=pd.DataFrame(summary);display(summary)
assert sha256(reservation_path)==protocol['reservation_sha256']
assert sha256(Q3_ZIP)==protocol['q3_zip_sha256']
assert sha256(score_path)==score_lock['matched_scores_sha256']
record={'created_utc':datetime.now(timezone.utc).isoformat(),'analysis_complete':True,
        'protocol_sha256':sha256(p),'score_lock_sha256':sha256(lock_path),
        'metrics_sha256':sha256(OUT/'matched_method_metrics.csv'),
        'matched_rows':len(paired),'comparison_method_count':len(methods),
        'failure_labels_used_for_thresholds':False,'failure_labels_used_for_evaluation':True,
        'q4_outcomes_opened':False,'sealed_June_test_opened':False,
        'winner_selected':False,'limitations':['Q3 aggregate outcomes previously inspected',
        'small complete-event sample; HGST may have no complete events',
        '30-day negative follow-up absent at Q3 end',
        'calibration quantile does not guarantee equal September alert burden',
        'Isolation Forest is a baseline; five-model masked Transformer is not evaluated here']}
(OUT/'matched_comparison_complete.json').write_text(json.dumps(record,indent=2))
print('NOTEBOOK 16 COMPLETE — matched Q3 detector and SMART-baseline comparison saved.')
print('Q4 unopened. Results:',OUT)

,horizon_days,evaluated_models,models_with_undefined_AP,complete_events_across_models
0,7,5,1,17
1,14,5,1,17
2,30,5,5,17


NOTEBOOK 16 COMPLETE — matched Q3 detector and SMART-baseline comparison saved.
Q4 unopened. Results: D:\SLIIT\Y4S1\RP\Research\J26-DS-342-\01_component_1_behavioral_drift_genai\results\single_model_q1q2_2021_v1\q3_matched_detector_baseline_comparison_v1
